In [1]:
from langchain_community.document_loaders import (
    TextLoader, 
    PyPDFLoader, 
    WebBaseLoader,
    DirectoryLoader
)
from langchain_core.documents import Document
import os

C:\Users\goran\AppData\Local\Temp\ipykernel_5936\3429779587.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [2]:
Document(page_content="This is a document", metadata={"source": "my_source"})

Document(metadata={'source': 'my_source'}, page_content='This is a document')

LangChain Document Processing Tutorial Components:
- TextLoader: Process text files for RAG with optimal chunk size
- PyPDFLoader: PDF to vector conversion with page-aware splitting
- WebBaseLoader: Web content processing for retrieval augmented generation
- DirectoryLoader: Batch document processing with text splitting strategies

In [3]:
from pathlib import Path

In [4]:
next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()) # look for something

WindowsPath('c:/Artificial Intelligence/rag_tutorial')

In [5]:
# Create sample files for testing

# Projektstamm anhand von pyproject.toml finden, unabhängig vom Arbeitsverzeichnis des Notebooks
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print(f"Project root: {project_root}")
documents_dir = project_root / "documents"
documents_dir.mkdir(exist_ok=True)
print(f"Documents dir: {documents_dir}")

Project root: c:\Artificial Intelligence\rag_tutorial
Documents dir: c:\Artificial Intelligence\rag_tutorial\documents


In [6]:
# 1. Loading Text Files
def load_text_documents(file_path: str)->list:
    """Load plain text files with proper encoding"""
    text_loader = TextLoader(str(file_path), encoding="utf-8")
    text_docs = text_loader.load()
    
    print(f"Loaded {len(text_docs)} text documents")
    print(f"First doc preview: {text_docs[0].page_content[:200]}...")
    print(f"Metadata: {text_docs[0].metadata}")
    
    return text_docs

In [7]:
# 2. Loading PDF Files
def load_pdf_documents(file_path: str)->list:
    """Load PDF files page by page"""
    pdf_loader = PyPDFLoader(str(file_path))
    pdf_docs = pdf_loader.load()
    
    print(f"Loaded {len(pdf_docs)} PDF pages")
    for i, doc in enumerate(pdf_docs[:3]):  # Show first 3 pages
        print(f"Page {i+1} preview: {doc.page_content[:150]}...")
        print(f"Page {i+1} metadata: {doc.metadata}")
    
    return pdf_docs

In [8]:
# 3. Loading Web Pages
def load_web_documents()->list:
    """Load and clean web content"""
    urls = [
        "https://python.langchain.com/docs/get_started/introduction",
        "https://python.langchain.com/docs/modules/data_connection"
    ]
    
    web_loader = WebBaseLoader(urls)
    web_docs = web_loader.load()
    
    print(f"Loaded {len(web_docs)} web pages")
    for doc in web_docs:
        print(f"URL: {doc.metadata.get('source', 'Unknown')}")
        print(f"Content preview: {doc.page_content[:200]}...")
    
    return web_docs

In [9]:
# 4. Batch Loading with DirectoryLoader
def load_directory_documents()->list:
    """Load all documents from a directory"""
    # pathlib-glob unterstützt keine {a,b}-Syntax, daher ein Loader pro Dateityp
    txt_loader = DirectoryLoader(
        str(documents_dir),
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"autodetect_encoding": True},
    )
    pdf_loader = DirectoryLoader(
        str(documents_dir),
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
    )

    docs = txt_loader.load() + pdf_loader.load()
    print(f"Loaded {len(docs)} documents from directory")

    # Group by file type
    file_types = {}
    for doc in docs:
        ext = doc.metadata["source"].split(".")[-1]
        file_types[ext] = file_types.get(ext, 0) + 1

    print(f"File types: {file_types}")
    return docs


In [10]:
# 5. Custom Loader with Error Handling
def load_documents_safely(file_paths: list)->tuple[list, list]:
    """Load documents with comprehensive error handling"""
    # Einzelnen Pfad (str oder Path) in eine Liste verpacken
    if isinstance(file_paths, (str, os.PathLike)): # wrap single path
        file_paths = [file_paths]

    loaded_docs = []
    failed_files = []
    
    for file_path in file_paths:
        file_path = str(file_path)
        try:
            if file_path.endswith('.pdf'):
                loader = PyPDFLoader(file_path)
            elif file_path.endswith('.txt'):
                loader = TextLoader(file_path, encoding='utf-8')
            else:
                print(f"Unsupported file type: {file_path}")
                continue
                
            docs = loader.load()
            loaded_docs.extend(docs)
            print(f"✅ Successfully loaded: {file_path}")
            
        except Exception as e:
            print(f"❌ Failed to load {file_path}: {str(e)}")
            failed_files.append((file_path, str(e)))
    
    print(f"\nSummary:")
    print(f"Successfully loaded: {len(loaded_docs)} documents")
    print(f"Failed files: {len(failed_files)}")
    
    if failed_files:
        print("\nFailed files:")
        for file_path, error in failed_files:
            print(f"  {file_path}: {error}")
    
    return loaded_docs, failed_files


In [11]:
# # Create sample files for testing
# from pathlib import Path

# # Projektstamm anhand von pyproject.toml finden, unabhängig vom Arbeitsverzeichnis des Notebooks
# project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
# print(f"Project root: {project_root}")
# documents_dir = project_root / "documents"
# documents_dir.mkdir(exist_ok=True)

with open(documents_dir / "sample_document.txt", "w", encoding="utf-8") as f:
    f.write("This is a sample document for testing LangChain loaders.\n" * 10)

In [12]:
# Test different loaders
print("Loading text documents...")
text_docs = load_text_documents(documents_dir / "sample_document.txt")
print("***"*20)
print(f"content: {text_docs[0].page_content[:100]}")
print(f"metadata: {text_docs[0].metadata}")
print(f"type: {type(text_docs[0])}")
print(text_docs[0].metadata)
print(f"Loaded {len(text_docs)} documents")

Loading text documents...
Loaded 1 text documents
First doc preview: This is a sample document for testing LangChain loaders.
This is a sample document for testing LangChain loaders.
This is a sample document for testing LangChain loaders.
This is a sample document for...
Metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\sample_document.txt'}
************************************************************
content: This is a sample document for testing LangChain loaders.
This is a sample document for testing LangC
metadata: {'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\sample_document.txt'}
type: <class 'langchain_core.documents.base.Document'>
{'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\sample_document.txt'}
Loaded 1 documents


In [13]:
# Clean up
sample_file = documents_dir / "sample_document.txt"
if sample_file.exists():
    print("Removing sample document")
    sample_file.unlink() # remove file

Removing sample document


In [14]:
 # For PDF, you'll need actual files
pdf_docs = load_pdf_documents(documents_dir / "ai-intro.pdf")
print("***"*20)
print(f"First page: {pdf_docs[0].page_content[:100]}")
print(f"Loaded {len(pdf_docs)} pages")


Loaded 10 PDF pages
Page 1 preview: Introduction to AI and ML
I. Definitions and Conceptual Foundations
Artificial Intelligence (AI) refers to the interdisciplinary scientific field devo...
Page 1 metadata: {'producer': 'pdfTeX-1.40.23', 'creator': 'LaTeX via pandoc', 'creationdate': '2026-02-02T23:01:43+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-02-02T23:01:43+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.23 (TeX Live 2021/W32TeX) kpathsea version 6.3.3', 'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}
Page 2 preview: P if its performance onT, as measured byP, improves withE.
ML functions as the dominant modern subfield of AI, having largely superseded
earlier exper...
Page 2 metadata: {'producer': 'pdfTeX-1.40.23', 'creator': 'LaTeX via pandoc', 'creationdate': '2026-02-02T23:01:43+01:00', 'author': '', 'title': '', '

In [15]:
# For PDF and web loading, you'll need actual files/URLs
web_docs = load_web_documents()

Loaded 2 web pages
URL: https://python.langchain.com/docs/get_started/introduction
Content preview: 




Quickstart - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main ...
URL: https://python.langchain.com/docs/modules/data_connection
Content preview: 




LangChain overview - Docs by LangChain









Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip ...


In [16]:
# Load all documents from directory
load_directory_documents()

Loaded 22 documents from directory
File types: {'txt': 10, 'pdf': 12}


[Document(metadata={'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.txt'}, page_content='# Introduction to AI and ML\n\n## I. Definitions and Conceptual Foundations\n\nArtificial Intelligence (AI) refers to the interdisciplinary scientific field\ndevoted to creating machines capable of performing tasks that typically require\nhuman intelligence. These tasks encompass perception (computer vision, speech\nrecognition), reasoning (logical inference, problem-solving), learning (pattern\nrecognition, adaptation), and decision-making under uncertainty. However, the\ndefinition of AI remains fluid; as computers master specific capabilitiesâ€”such\nas arithmetic, chess, or facial recognitionâ€”these tasks often cease to be\nclassified as "artificial intelligence" in popular discourse and instead become\nmere "computation." This phenomenon, known as the **AI effect**, suggests that\nintelligence is frequently defined as whatever machines cannot yet do.\n\nWithin AI res

In [17]:
# sample_document.txt wurde zuvor gelöscht, daher eine vorhandene Datei verwenden
load_documents_safely(documents_dir / "ai-intro.pdf")


✅ Successfully loaded: c:\Artificial Intelligence\rag_tutorial\documents\ai-intro.pdf

Summary:
Successfully loaded: 10 documents
Failed files: 0


([Document(metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'LaTeX via pandoc', 'creationdate': '2026-02-02T23:01:43+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-02-02T23:01:43+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.23 (TeX Live 2021/W32TeX) kpathsea version 6.3.3', 'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\ai-intro.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1'}, page_content='Introduction to AI and ML\nI. Definitions and Conceptual Foundations\nArtificial Intelligence (AI) refers to the interdisciplinary scientific field devoted to\ncreating machines capable of performing tasks that typically require human intel-\nligence. These tasks encompass perception (computer vision, speech recognition),\nreasoning (logical inference, problem-solving), learning (pattern recognition,\nadaptation), and decision-making under uncertainty. However, the definition of\nAI remain

In [18]:
load_documents_safely(documents_dir / "python-intro.pdf")

✅ Successfully loaded: c:\Artificial Intelligence\rag_tutorial\documents\python-intro.pdf

Summary:
Successfully loaded: 2 documents
Failed files: 0


([Document(metadata={'producer': 'pdfTeX-1.40.23', 'creator': 'LaTeX via pandoc', 'creationdate': '2026-02-02T23:02:54+01:00', 'author': '', 'title': '', 'subject': '', 'keywords': '', 'moddate': '2026-02-02T23:02:54+01:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.23 (TeX Live 2021/W32TeX) kpathsea version 6.3.3', 'source': 'c:\\Artificial Intelligence\\rag_tutorial\\documents\\python-intro.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Python: A Versatile Programming Language\nPython stands as one of the most popular programming languages worldwide,\ndistinguished by its emphasis on code readability and developer productivity.\nCreated by Guido van Rossum and first released in 1991, Python has evolved\nfrom a modest scripting language into a powerhouse driving modern software\ndevelopment, data science, and artificial intelligence.\nHistorical Evolution\nPython’s development began in the late 1980s at the Centrum Wisk